# Session 11 — Spark & Databricks: Hello World
**Big Data Analytics · QuickKart lab**

Run this top to bottom on any Databricks cluster (Community Edition is fine). No file upload needed — this notebook generates its own small QuickKart orders table so it works standalone.

Five things this notebook proves, in order:
1. Spark is already there — no setup
2. Read data into a DataFrame
3. Ask a business question with the four verbs (filter → group → aggregate → sort)
4. See that transformations are lazy, and watch the DAG with `.explain()`
5. Feel the difference caching makes on repeated, iterative work

## 1 — Spark is already there
Databricks pre-creates a `SparkSession` called `spark` in every notebook. Nothing to import, nothing to configure.

In [ ]:
spark

In [ ]:
# Quick check — how many worker cores does this cluster actually have?
sc = spark.sparkContext
print("App name:", sc.appName)
print("Default parallelism (cores available):", sc.defaultParallelism)

## 2 — Build the QuickKart orders table
In a real class you'd read this from `/FileStore/quickkart_orders.csv` on DBFS. Here we generate ~200,000 synthetic rows in-memory so the notebook runs anywhere, instantly — same shape of data, same columns you'd see in the real file.

In [ ]:
import random
from pyspark.sql import Row
from pyspark.sql import functions as F

random.seed(42)

categories = ["Fruits & Veg", "Dairy", "Snacks", "Beverages", "Household", "Personal Care"]
regions    = ["North", "South", "East", "West"]
N = 200_000

def make_row(i):
    is_festival = random.random() < 0.15
    return Row(
        order_id   = f"QK-{100000+i}",
        category   = random.choice(categories),
        region     = random.choice(regions),
        revenue    = round(random.uniform(50, 1500), 2),
        festival_week = is_festival,
    )

rows = [make_row(i) for i in range(N)]
df = spark.createDataFrame(rows)
df.printSchema()

In [ ]:
# Cell counts as an ACTION -> this is the first time Spark actually runs anything above
print(f"{df.count():,} orders generated")
df.show(5)

## 3 — Ask a business question: the four verbs
Same shape as the Hive lab — **filter, group, aggregate, sort** — just on the faster engine. This mirrors the "festival week revenue by category" example from the slide deck's DAG diagram.

In [ ]:
result = (
    df.filter(F.col("festival_week") == True)
      .groupBy("category")
      .agg(F.sum("revenue").alias("total_revenue"))
      .orderBy(F.desc("total_revenue"))
)
result.show()

## 4 — Transformations are lazy: prove it to yourself
Nothing above actually "ran" until an **action** (`.show()`, `.count()`) was called. Build a chain of transformations and inspect the plan *before* triggering it.

In [ ]:
# Build the plan only -- no action called yet, so nothing executes
plan_only = (
    df.filter(F.col("region") == "West")
      .groupBy("category")
      .sum("revenue")
)
print(type(plan_only))   # still just a DataFrame -- a description of work, not a result
print("No output above from Spark itself -- because nothing has run yet.")

In [ ]:
# NOW trigger it, and look at the DAG Spark actually built
plan_only.explain()

Read the output above bottom-up: `Scan` (the read) happens first, then `Filter`, then the shuffle for `groupBy`, then the aggregation. This is the DAG from the slide deck, printed straight from the engine — a job broken into stages, each stage a set of tasks.

## 5 — Caching: feel the difference on iterative work
This is the exact QuickKart problem from the slides: Meera wants to test **several scenarios** against the same base table. Compare an uncached vs a cached run.

In [ ]:
import time

# Uncached: every action below re-reads/re-computes from scratch
uncached = df.filter(F.col("revenue") > 500)

t0 = time.time()
uncached.groupBy("region").count().collect()
t1 = time.time()
uncached.groupBy("category").count().collect()
t2 = time.time()

print(f"Run 1 (uncached): {t1 - t0:.2f}s")
print(f"Run 2 (uncached): {t2 - t1:.2f}s")

In [ ]:
# Cached: computed once, reused for every scenario after that
cached = df.filter(F.col("revenue") > 500).cache()
cached.count()  # action that materialises the cache

t0 = time.time()
cached.groupBy("region").count().collect()
t1 = time.time()
cached.groupBy("category").count().collect()
t2 = time.time()

print(f"Run 1 (cached, warming up): {t1 - t0:.2f}s")
print(f"Run 2 (cached, reused):     {t2 - t1:.2f}s")
print("\nOn a real multi-scenario job with 5+ what-if queries, this gap is the")
print("difference between Meera's 9 AM meeting having numbers or not.")

## Change-one-thing exercises
Do these in order. Each is a one-line edit to a cell above.

1. **Change the filter.** In section 3, change `festival_week == True` to `festival_week == False`. Does total revenue by category change much? What does that tell you about festival-week spending patterns?
2. **Change the group.** In section 3, group by `region` instead of `category`. Which region wins?
3. **Break the cache.** In section 5, comment out `.cache()` and re-run both timed cells. Do the two "cached" runs still get faster on the second call? Why or why not?
4. **Read `.explain()` again.** After adding `.cache()`, call `.explain()` on the cached DataFrame. Do you see `InMemoryTableScan` anywhere? That's Spark telling you it's reading from memory, not from the original data.

## One-line summary
`spark` was already there. Reading data, filtering, grouping, and sorting is the same four-verb shape as the Hive lab. Nothing runs until an action is called — and `.cache()` is how you stop Spark from redoing the same work for every follow-up question.